# Task 2.1 — Prompt Chaining & Question Decomposition

Questo notebook implementa una pipeline avanzata di **Prompt Chaining** per risolvere il problema delle *multi-barrelled questions* (domande composte o multiple), uno dei limiti principali evidenziati dalla letteratura nell'analisi automatica del discorso politico.

Quando un giornalista pone più domande all'interno dello stesso turno di parola, i modelli linguistici faticano a fare il *grounding* corretto della risposta. Se un politico risponde chiaramente a una parte della domanda ma ignora le altre, un approccio standard rischia di etichettare l'intera risposta come `Explicit` o `Dodging`, perdendo le sfumature.

Per superare questo limite, utilizziamo il modello **Llama 3.1 8B-Instruct** (precedentemente sottoposto a fine-tuning nel Task 2 per riconoscere le 9 tecniche di evasione) come "motore logico" all'interno di un'architettura a 3 passaggi:

1. **Step 1 (Splitter):** Il modello analizza la domanda originale del giornalista e, se individua più interrogativi, li scompone in singole sotto-domande (sQA) atomiche restituite in formato JSON.
2. **Step 2 (Evaluator):** Il modello valuta la risposta globale del politico confrontandola iterativamente con ciascuna delle sotto-domande generate, assegnando a ognuna una delle 9 tecniche di evasione (es. *Explicit*, *Deflection*, *Dodging*).
3. **Step 3 (Aggregator):** Un algoritmo deterministico analizza le valutazioni multiple. Se rileva che il politico ha risposto in modo esplicito ad almeno una sotto-domanda, ma ha evaso le altre, applica un'etichetta aggregata di **Partial/half-answer**, mappando infine il risultato sulle 3 macro-categorie di chiarezza.

*Nota: Questo notebook non esegue addestramento, ma esegue inferenza complessa caricando i pesi (adapter LoRA) salvati al termine del Task 2.*

In [ ]:
# ============================================================
# CELLA 1 — Setup & Installazione
# ============================================================

# !pip install -q -U transformers peft trl bitsandbytes datasets accelerate pyarrow

import os, sys, json, gc
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_from_disk
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

# Mount Google Drive
try:
    import google.colab
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=True)

    # Login HF
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Colab configurato con successo.")

except ImportError:
    print("Ambiente locale rilevato.")

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ============================================================
# CELLA 2 — Configurazione, Percorsi e Prompt
# ============================================================

# ── Modello e percorsi ────────────────────────────────────────
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# Carichiamo i pesi addestrati del Task 2 (assicurati che il path sia corretto)
PATH_MODELLO_TASK2 = "/content/drive/MyDrive/progettoLLM/risultati_task2/modello_finale"

# Nuova cartella per i risultati del Prompt Chaining
RISULTATI_DIR = "/content/drive/MyDrive/progettoLLM/risultati_task_chaining"

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# ── PROMPT 1: Lo Splitter (Decomposizione Domanda) ────────────
PROMPT_SPLITTER = """Sei un assistente linguistico esperto. Analizza la seguente domanda di un'intervista giornalistica.
Se la domanda contiene più sotto-domande separate, dividile in una lista.
Se è una domanda singola, restituiscila come lista di un solo elemento.

Rispondi ESCLUSIVAMENTE in un formato JSON strutturato in questo modo:
{"sub_questions": ["domanda 1", "domanda 2"]}
Non aggiungere formattazione markdown, né altro testo.
"""

# ── PROMPT 2: L'Evaluator (Le 9 Categorie del Task 2) ─────────
PROMPT_EVALUATOR = """Sei un esperto analista di comunicazione politica. Classifica la risposta del politico alla domanda scegliendo ESATTAMENTE UNA delle seguenti 9 categorie:

1. 'Explicit'
2. 'Declining to answer'
3. 'Claims ignorance'
4. 'Dodging'
5. 'Deflection'
6. 'General'
7. 'Implicit'
8. 'Partial/half-answer'
9. 'Clarification'

Rispondi ESCLUSIVAMENTE fornendo un oggetto JSON valido contenente la singola chiave "categoria". Esempio: {"categoria": "Deflection"}. Non aggiungere altro testo.
"""

# ── Mapping alle 3 Macro-Categorie ────────────────────────────
MAPPING_9_CLASSI = {
    "Explicit": "Clear Reply",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Dodging": "Ambivalent",
    "Deflection": "Ambivalent",
    "General": "Ambivalent",
    "Implicit": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Clarification": "Ambivalent"
}

print("Configurazione caricata.")

In [ ]:
# ============================================================
# CELLA 3 — Dataset Management
# ============================================================

# Per questo task ci serve solo il test set per valutare la pipeline
test_dataset_path = "/content/drive/MyDrive/progettoLLM/CLARITY/dataset/QEvasion/test"
test_dataset = load_from_disk(test_dataset_path)

print(f"Test Set caricato: {len(test_dataset)} esempi.")

In [ ]:
# ============================================================
# CELLA 4 — Caricamento Modello Fine-Tuned (Task 2)
# ============================================================

torch.cuda.empty_cache()
gc.collect()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

print(f"Caricamento modello base ({MODEL_ID}) in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    device_map="auto",
    quantization_config=bnb_config, 
    torch_dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True
)

print(f"Applicazione degli adattatori del Task 2 da: {PATH_MODELLO_TASK2}")
model = PeftModel.from_pretrained(
    base_model, PATH_MODELLO_TASK2,
    low_cpu_mem_usage=True, device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(PATH_MODELLO_TASK2)
model.eval()

print(f"Motore di inferenza pronto! Device: {model.device}")

In [ ]:
# ============================================================
# CELLA 5 — Pipeline Multi-Step (Prompt Chaining)
# ============================================================

def chiedi_all_llm(system_prompt, user_content, max_tokens=150):
    """Funzione universale per interrogare l'LLM e ricevere un JSON."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )

    raw_output = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    clean_json = raw_output.replace("```json", "").replace("```", "").strip()
    
    try:
        return json.loads(clean_json)
    except json.JSONDecodeError:
        return None

def split_domanda(domanda):
    """Step 1: Divide la domanda complessa in sotto-domande."""
    risultato = chiedi_all_llm(PROMPT_SPLITTER, f"Domanda da analizzare: {domanda}", max_tokens=200)
    if risultato and "sub_questions" in risultato:
        return risultato["sub_questions"]
    return [domanda]

def valuta_sotto_domanda(sotto_domanda, risposta_globale):
    """Step 2: Valuta la risposta contro una singola sotto-domanda."""
    user_content = f"Domanda specifica: {sotto_domanda}\nRisposta globale: {risposta_globale}"
    risultato = chiedi_all_llm(PROMPT_EVALUATOR, user_content, max_tokens=30)
    if risultato and "categoria" in risultato:
        return risultato["categoria"]
    return "Dodging"

def inferenza_chaining(example):
    """Step 3: Orchestra la pipeline e aggrega i risultati."""
    domanda = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')
    
    sotto_domande = split_domanda(domanda)
    
    valutazioni = []
    for sq in sotto_domande:
        valutazioni.append(valuta_sotto_domanda(sq, risposta))
        
    # LOGICA DI AGGREGAZIONE (Partial/half-answer detection)
    if len(sotto_domande) == 1:
        tecnica_finale = valutazioni[0]
    else:
        ha_risposto = "Explicit" in valutazioni
        ha_evaso = any(val != "Explicit" for val in valutazioni)
        
        if ha_risposto and ha_evaso:
            tecnica_finale = "Partial/half-answer"
        elif not ha_evaso:
            tecnica_finale = "Explicit"
        else:
            tecnica_finale = Counter(valutazioni).most_common(1)[0][0]
            
    # Fallback di sicurezza se genera roba strana
    if tecnica_finale not in MAPPING_9_CLASSI:
        tecnica_finale = "Dodging"
        
    macro_clarity = MAPPING_9_CLASSI[tecnica_finale]
    
    return tecnica_finale, macro_clarity, sotto_domande, valutazioni

In [ ]:
# ============================================================
# CELLA 6 — Valutazione Definitiva
# ============================================================

y_true_clarity = []
y_pred_clarity = []
y_pred_tecnica = []
log_scomposizioni = []

print(f"Avvio Prompt Chaining su {len(test_dataset)} esempi (Richiederà tempo)...")

# Usa un numero più piccolo per fare un test iniziale, es: range(10)
for i in tqdm(range(len(test_dataset))):
    esempio = test_dataset[i]
    y_true_clarity.append(str(esempio.get('clarity_label', '')).strip())

    tecnica, clarity, sub_q, sub_evals = inferenza_chaining(esempio)
    
    y_pred_tecnica.append(tecnica)
    y_pred_clarity.append(clarity)
    log_scomposizioni.append({"sub_questions": sub_q, "evaluations": sub_evals})

# ── Report e Grafici ─────────────────────────────────────────
LABELS = ["Ambivalent", "Clear Reply", "Clear Non-Reply"]

print("\n" + "=" * 60)
print("REPORT FINALE — TASK 3 (PROMPT CHAINING)")
print("=" * 60)
print(classification_report(y_true_clarity, y_pred_clarity, labels=LABELS, digits=3))

os.makedirs(RISULTATI_DIR, exist_ok=True)

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true_clarity, y_pred_clarity, labels=LABELS)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predizioni (Prompt Chaining)')
plt.ylabel('Etichette Reali')
plt.title('Matrice di Confusione — Prompt Chaining Pipeline')
plt.tight_layout()
plt.savefig(f"{RISULTATI_DIR}/matrice_confusione_chaining.png", bbox_inches='tight', dpi=300)
plt.show()

# ── Salvataggio Risultati Avanzati in CSV ───────────────────
df_risultati = pd.DataFrame({
    'Domanda_Originale': [ex.get('interview_question', '') for ex in test_dataset],
    'Risposta_Politico': [ex.get('interview_answer', '') for ex in test_dataset],
    'Sotto_Domande': [str(log["sub_questions"]) for log in log_scomposizioni],
    'Valutazioni_Singole': [str(log["evaluations"]) for log in log_scomposizioni],
    'Tecnica_Aggregata': y_pred_tecnica,
    'Predetto_Macro_Clarity': y_pred_clarity,
    'Vero_Macro_Clarity': y_true_clarity
})

percorso_csv = f"{RISULTATI_DIR}/risultati_prompt_chaining.csv"
df_risultati.to_csv(percorso_csv, index=False)
print(f"CSV con log di scomposizione salvato in: {percorso_csv}")